In [1]:
"""Stufe 2: Statistisches Modell je Reihe (exponentielle Glättung mit Wochenmuster), Bericht 3.5 und 3.7.
Ausführen: uv run python src/statistical.py   (Ausgabe: reports/statistical_output.txt, reports/results_statistical.csv)
Vorher: src/clean.py und src/baselines.py (für den Benchmark) müssen gelaufen sein.
"""

'Stufe 2: Statistisches Modell je Reihe (exponentielle Glättung mit Wochenmuster), Bericht 3.5 und 3.7.\nAusführen: uv run python src/statistical.py   (Ausgabe: reports/statistical_output.txt, reports/results_statistical.csv)\nVorher: src/clean.py und src/baselines.py (für den Benchmark) müssen gelaufen sein.\n'

In [2]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
sys.path.insert(0, str(ROOT / "src"))
from forecast_eval import WINDOWS, load_clean, score, seasonal_baselines, to_wide, window_frames  # noqa: E402
from stat_models import ets_forecast  # noqa: E402

pd.set_option("display.width", 170)
lines = []


def out(msg=""):
    print(msg)
    lines.append(str(msg))


df = load_clean()
W_train = to_wide(df, "sales_train")

## 1. Das Modell
Für **jede** der 1.782 Reihen wird einzeln ein Holt-Winters-Modell angepasst (statsmodels `ExponentialSmoothing`):
- Zielgröße `log1p(Absatz)`, danach Rücktransformation mit `expm1` und Begrenzung auf mindestens 0
- **Level** (aktuelles Niveau, jüngere Tage zählen mehr) plus **additives Wochenmuster** (7 Tage), kein Trend
- Historie: die letzten 365 Tage bis zum Prognose-Ursprung, maskierte Tage linear überbrückt
- **Rückfall auf die Baseline** (Mittel 4 Wochen), wenn eine Reihe weniger als 28 verwertbare Tage hat oder ihr
  letzter Wert mehr als 14 Tage vor dem Ursprung liegt

Das Modell sieht keine Promotion, keine Feiertage, keine anderen Reihen. Es ist rein statistisch je Reihe.

In [3]:
rows, per_family, fallback_counts = [], {}, {}
for name, origin in WINDOWS.items():
    truth, excluded, outlier = window_frames(df, origin)
    b2 = seasonal_baselines(W_train.loc[:origin], origin)["B2 Mittel 4 Wochen"]
    pred, counts = ets_forecast(W_train, origin, fallback=b2)
    r = score(truth, pred, excluded, outlier)
    rows.append({"Fenster": name, "Modell": "ETS je Reihe", "RMSLE": r["rmsle"], "RMSLE ohne Spitzen": r["rmsle_ohne_spitzen"],
                 "bewertet": r["n_bewertet"], "ohne Prognose": r["n_ohne_prognose"]})
    per_family[name] = r["je_familie"]
    fallback_counts[name] = counts
res = pd.DataFrame(rows)
out(res.round(4).to_string(index=False))
out("\nReihen je Fenster: mit Modell / Rückfall zu wenig Daten / Rückfall zu alt / Fit-Fehler / gar keine Prognose")
for k, c in fallback_counts.items():
    out(f"{k}: {c['modell']} / {c['zu_wenig_daten']} / {c['zu_alt']} / {c['fit_fehler']} / {c['ohne_prognose']}")

    Fenster       Modell  RMSLE  RMSLE ohne Spitzen  bewertet  ohne Prognose
W1 Aug 2017 ETS je Reihe 0.4845              0.4844     28512              0
W2 Dez 2016 ETS je Reihe 0.5512              0.5494     27981              3
W3 Mär 2017 ETS je Reihe 0.4108              0.4105     27984              0

Reihen je Fenster: mit Modell / Rückfall zu wenig Daten / Rückfall zu alt / Fit-Fehler / gar keine Prognose
W1 Aug 2017: 1782 / 0 / 0 / 0 / 0
W2 Dez 2016: 1748 / 34 / 0 / 0 / 33
W3 Mär 2017: 1749 / 33 / 0 / 0 / 33


## 2. Vergleich mit dem Benchmark
Benchmark = beste Baseline je Fenster (Mittel der letzten 4 gleichen Wochentage). Erfolgskriterium (Bericht 3.4):
10 % niedrigerer RMSLE im Mittel der drei Fenster und in mindestens zwei Fenstern einzeln.

In [4]:
base = pd.read_csv(ROOT / "reports" / "results_baselines.csv")
bench = base.pivot(index="Fenster", columns="Modell", values="RMSLE").min(axis=1).reindex(list(WINDOWS))
ets = res.set_index("Fenster").RMSLE
cmp_ = pd.DataFrame({"Benchmark": bench, "ETS je Reihe": ets})
cmp_["Änderung"] = (cmp_["ETS je Reihe"] / cmp_.Benchmark - 1).map("{:+.1%}".format)
cmp_["Ziel (-10 %)"] = (0.9 * cmp_.Benchmark).round(4)
cmp_["erreicht"] = cmp_["ETS je Reihe"] <= cmp_["Ziel (-10 %)"]
out("\n" + cmp_.round(4).to_string())
mean_b, mean_e = bench.mean(), ets.mean()
out(f"\nMittel: Benchmark {mean_b:.4f}, ETS {mean_e:.4f}, Änderung {mean_e / mean_b - 1:+.1%} (Ziel: {0.9 * mean_b:.4f})")
out(f"Erfolgskriterium: Mittel <= Ziel: {mean_e <= 0.9 * mean_b}; Fenster einzeln erreicht: {int(cmp_['erreicht'].sum())} von 3")


             Benchmark  ETS je Reihe Änderung  Ziel (-10 %)  erreicht
Fenster                                                              
W1 Aug 2017     0.5306        0.4845    -8.7%        0.4775     False
W2 Dez 2016     0.5938        0.5512    -7.2%        0.5345     False
W3 Mär 2017     0.4523        0.4108    -9.2%        0.4071     False

Mittel: Benchmark 0.5256, ETS 0.4822, Änderung -8.3% (Ziel: 0.4730)
Erfolgskriterium: Mittel <= Ziel: False; Fenster einzeln erreicht: 0 von 3


## 3. Wo gewinnt, wo verliert das Modell? (Fehler je Familie, W1)

In [5]:
truth, excluded, outlier = window_frames(df, WINDOWS["W1 Aug 2017"])
b2_w1 = seasonal_baselines(W_train.loc[:WINDOWS["W1 Aug 2017"]], WINDOWS["W1 Aug 2017"])["B2 Mittel 4 Wochen"]
base_fam = score(truth, b2_w1, excluded, outlier)["je_familie"]
fam = pd.DataFrame({"B2 Baseline": base_fam, "ETS": per_family["W1 Aug 2017"]})
fam["Änderung"] = fam.ETS / fam["B2 Baseline"] - 1
out("\nFamilien in W1, größte Verbesserung zuerst:\n" + fam.sort_values("Änderung").assign(
    Änderung=lambda d: d.Änderung.map("{:+.1%}".format)).round(3).to_string())
out(f"\nFamilien mit besserem ETS: {(fam.ETS < fam['B2 Baseline']).sum()} von {len(fam)}")


Familien in W1, größte Verbesserung zuerst:
                            B2 Baseline    ETS Änderung
HOME APPLIANCES                   0.405  0.314   -22.4%
BOOKS                             0.101  0.081   -19.5%
BABY CARE                         0.336  0.281   -16.4%
SCHOOL AND OFFICE SUPPLIES        1.799  1.535   -14.7%
HARDWARE                          0.603  0.521   -13.6%
LADIESWEAR                        0.538  0.479   -11.1%
AUTOMOTIVE                        0.581  0.523   -10.0%
PLAYERS AND ELECTRONICS           0.512  0.461    -9.9%
SEAFOOD                           0.564  0.509    -9.8%
PET SUPPLIES                      0.522  0.473    -9.3%
HOME AND KITCHEN II               0.512  0.470    -8.2%
EGGS                              0.329  0.303    -8.0%
BEVERAGES                         0.270  0.250    -7.5%
HOME AND KITCHEN I                0.525  0.490    -6.7%
LAWN AND GARDEN                   0.464  0.434    -6.5%
MAGAZINES                         0.576  0.539    -6.4%
CEL

## 4. Empfindlichkeit: wie viel Historie?
Die 365 Tage haben wir gewählt, nicht gemessen. Zum Vergleich 90 und 180 Tage.
Achtung: Wir wählen hier anhand derselben Fenster, mit denen wir bewerten. Bei nur drei Varianten ist das tolerierbar,
aber der Unterschied sollte größer als das Rauschen sein, bevor wir daraus etwas schließen.

In [6]:
sens = {}
for hd in (90, 180, 365):
    vals = []
    for name, origin in WINDOWS.items():
        truth, excluded, outlier = window_frames(df, origin)
        b2 = seasonal_baselines(W_train.loc[:origin], origin)["B2 Mittel 4 Wochen"]
        pred, _ = ets_forecast(W_train, origin, fallback=b2, history_days=hd)
        vals.append(score(truth, pred, excluded, outlier)["rmsle"])
    sens[f"{hd} Tage"] = vals
sens = pd.DataFrame(sens, index=list(WINDOWS))
sens.loc["Mittel"] = sens.mean()
out("\nRMSLE je Historienlänge:\n" + sens.round(4).to_string())


RMSLE je Historienlänge:
             90 Tage  180 Tage  365 Tage
W1 Aug 2017   0.4974    0.4943    0.4845
W2 Dez 2016   0.5677    0.5605    0.5512
W3 Mär 2017   0.4238    0.4125    0.4108
Mittel        0.4963    0.4891    0.4822


In [7]:
res.to_csv(ROOT / "reports" / "results_statistical.csv", index=False)
(ROOT / "reports" / "statistical_output.txt").write_text("\n".join(lines), encoding="utf-8")

3235